# 04 — Camada Gold

## Objetivo

Nesta etapa vamos transformar os dados tratados da camada Silver em um modelo preparado para análise e Business Intelligence.

Utilizaremos uma estrutura chamada **Star Schema (Modelo Estrela)**.

O modelo será composto por:

### Dimensões

- `dim_data`
- `dim_produto`
- `dim_cliente`
- `dim_regiao`

### Tabela Fato

- `fato_vendas`

As dimensões armazenam informações utilizadas para descrever e categorizar as vendas.

A tabela fato armazenará os eventos de venda e suas métricas.

Nosso fluxo será:

`silver.vendas → modelagem dimensional → Gold → Apache Superset`

## Etapa 1 — Importação das bibliotecas

Neste notebook utilizaremos o Python principalmente para controlar a execução do nosso pipeline.

As transformações da camada Gold serão realizadas utilizando SQL diretamente no PostgreSQL.

Utilizaremos:

- `psycopg2`: comunicação entre Python e PostgreSQL;
- `getpass`: entrada segura da senha do banco.

In [ ]:
# ==========================================================
# ETAPA 1 - IMPORTAÇÃO DAS BIBLIOTECAS
# ==========================================================

# Biblioteca utilizada para conectar
# Python ao PostgreSQL.
import psycopg2


# Biblioteca utilizada para solicitar a senha
# sem deixá-la visível no notebook.
from getpass import getpass

## Etapa 2 — Configuração da conexão

Vamos configurar novamente os dados necessários para acessar nosso PostgreSQL.

O banco continua executando através do Docker.

A conexão utilizada será:

`Python → localhost:5434 → Docker → PostgreSQL → db_vendas`

In [ ]:
# ==========================================================
# ETAPA 2 - CONFIGURAÇÃO DA CONEXÃO
# ==========================================================

# Endereço do PostgreSQL.
DB_HOST = "localhost"


# Porta externa configurada no Docker.
DB_PORT = 5434


# Database utilizado pelo projeto.
DB_NAME = "db_vendas"


# Usuário PostgreSQL.
DB_USER = "postgres"


# Solicita a senha sem deixá-la escrita
# diretamente no notebook.
DB_PASSWORD = getpass("Digite a senha do PostgreSQL: ")

## Etapa 3 — Teste de conexão

Antes de iniciar a construção da camada Gold, vamos verificar se conseguimos acessar o PostgreSQL.

Essa verificação evita iniciar o pipeline caso exista algum problema de conexão.

In [ ]:
# ==========================================================
# ETAPA 3 - TESTE DE CONEXÃO
# ==========================================================

try:

    # Abre uma conexão com o PostgreSQL.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )


    print("✅ Conexão com PostgreSQL realizada com sucesso!")


except Exception as erro:

    print("❌ Não foi possível conectar ao PostgreSQL.")
    print("Tipo do erro:", type(erro).__name__)


finally:

    # Fecha a conexão após o teste.
    if "conexao" in locals():
        conexao.close()

## Etapa 4 — Criação do Schema Gold

Já temos dois schemas em nosso banco:

- `bronze`: dados recebidos da fonte;
- `silver`: dados tratados e padronizados.

Agora criaremos:

- `gold`: dados modelados para análise.

Dentro do schema Gold construiremos nosso modelo dimensional.

Estrutura planejada:

gold
- dim_data
- dim_produto
- dim_cliente
- dim_regiao
- fato_vendas

In [ ]:
# ==========================================================
# ETAPA 4 - CRIAÇÃO DO SCHEMA GOLD
# ==========================================================

try:

    # Abre a conexão com o PostgreSQL.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )


    # Cria o cursor responsável por
    # executar os comandos SQL.
    cursor = conexao.cursor()


    # Cria o schema Gold.
    #
    # IF NOT EXISTS evita erro caso o schema
    # já tenha sido criado anteriormente.
    cursor.execute("""
        CREATE SCHEMA IF NOT EXISTS gold;
    """)


    # Confirma a alteração no PostgreSQL.
    conexao.commit()


    print("✅ Schema gold criado com sucesso!")


except Exception as erro:

    # Caso ocorra algum problema,
    # desfaz a transação.
    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro ao criar o schema Gold.")
    print("Tipo do erro:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    # Fecha o cursor.
    if "cursor" in locals():
        cursor.close()

    # Fecha a conexão.
    if "conexao" in locals():
        conexao.close()

## Etapa 5 — Criação da Dimensão Data

A primeira dimensão do nosso modelo será:

`gold.dim_data`

Ela armazenará informações relacionadas às datas existentes nas vendas.

Cada data aparecerá apenas uma vez na dimensão.

Exemplo:

Se existirem 50 vendas realizadas em `05/01/2026`, a dimensão continuará possuindo apenas um registro referente a essa data.

A tabela fato utilizará posteriormente a chave `id_data` para se relacionar com essa dimensão.

A dimensão terá:

- `id_data`: chave da dimensão;
- `data`: data completa;
- `dia`: número do dia;
- `mes`: número do mês;
- `nome_mes`: nome do mês;
- `ano`: ano;
- `trimestre`: trimestre do ano;
- `dia_semana`: nome do dia da semana.

In [ ]:
# ==========================================================
# ETAPA 6 - CRIAÇÃO DA DIMENSÃO DATA
# ==========================================================

try:

    # Abre a conexão com o PostgreSQL.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )


    # Cria o cursor para executar SQL.
    cursor = conexao.cursor()


    # Cria a tabela dim_data.
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS gold.dim_data (

            -- Chave primária da dimensão.
            id_data SERIAL PRIMARY KEY,

            -- Data completa da venda.
            data DATE NOT NULL UNIQUE,

            -- Número do dia.
            dia INTEGER NOT NULL,

            -- Número do mês.
            mes INTEGER NOT NULL,

            -- Nome do mês.
            nome_mes TEXT NOT NULL,

            -- Ano.
            ano INTEGER NOT NULL,

            -- Número do trimestre.
            trimestre INTEGER NOT NULL,

            -- Nome do dia da semana.
            dia_semana TEXT NOT NULL
        );
    """)


    # Confirma a criação da tabela.
    conexao.commit()


    print("✅ gold.dim_data criada com sucesso!")


except Exception as erro:

    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro ao criar gold.dim_data.")
    print("Tipo:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 7 — Carga da Dimensão Data

Agora que a estrutura da `gold.dim_data` foi criada, vamos carregar os dados.

A fonte será:

`silver.vendas`

Como uma mesma data pode aparecer em várias vendas, utilizaremos `DISTINCT` para selecionar somente datas diferentes.

A partir da coluna `data_venda`, criaremos:

- dia;
- mês;
- nome do mês;
- ano;
- trimestre;
- dia da semana.

O PostgreSQL será responsável por realizar essas transformações.

In [ ]:
# ==========================================================
# ETAPA 7 - CARGA DA DIMENSÃO DATA
# ==========================================================

try:

    # ------------------------------------------------------
    # 1. ABRIR CONEXÃO
    # ------------------------------------------------------

    # Abre uma conexão com nosso banco PostgreSQL.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )


    # Cria o cursor responsável por executar SQL.
    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 2. LIMPAR A DIMENSÃO
    # ------------------------------------------------------

    # Remove os registros existentes na dim_data.
    #
    # RESTART IDENTITY faz o contador SERIAL voltar para 1.
    #
    # Assim podemos executar novamente esta etapa
    # sem duplicar registros.
    cursor.execute("""
        TRUNCATE TABLE gold.dim_data
        RESTART IDENTITY;
    """)


    # ------------------------------------------------------
    # 3. INSERIR AS DATAS
    # ------------------------------------------------------

    cursor.execute("""
        INSERT INTO gold.dim_data (
            data,
            dia,
            mes,
            nome_mes,
            ano,
            trimestre,
            dia_semana
        )

        SELECT DISTINCT

            -- Data completa.
            data_venda,


            -- Extrai o número do dia.
            EXTRACT(DAY FROM data_venda)::INTEGER,


            -- Extrai o número do mês.
            EXTRACT(MONTH FROM data_venda)::INTEGER,


            -- Converte o número do mês para o nome
            -- correspondente em português.
            CASE EXTRACT(MONTH FROM data_venda)

                WHEN 1 THEN 'Janeiro'
                WHEN 2 THEN 'Fevereiro'
                WHEN 3 THEN 'Março'
                WHEN 4 THEN 'Abril'
                WHEN 5 THEN 'Maio'
                WHEN 6 THEN 'Junho'
                WHEN 7 THEN 'Julho'
                WHEN 8 THEN 'Agosto'
                WHEN 9 THEN 'Setembro'
                WHEN 10 THEN 'Outubro'
                WHEN 11 THEN 'Novembro'
                WHEN 12 THEN 'Dezembro'

            END,


            -- Extrai o ano.
            EXTRACT(YEAR FROM data_venda)::INTEGER,


            -- Extrai o trimestre.
            EXTRACT(QUARTER FROM data_venda)::INTEGER,


            -- Converte o número do dia da semana
            -- para seu nome em português.
            CASE EXTRACT(DOW FROM data_venda)

                WHEN 0 THEN 'Domingo'
                WHEN 1 THEN 'Segunda-feira'
                WHEN 2 THEN 'Terça-feira'
                WHEN 3 THEN 'Quarta-feira'
                WHEN 4 THEN 'Quinta-feira'
                WHEN 5 THEN 'Sexta-feira'
                WHEN 6 THEN 'Sábado'

            END

        FROM silver.vendas

        -- Evita inserir uma eventual data nula.
        WHERE data_venda IS NOT NULL

        -- Organiza a inserção cronologicamente.
        ORDER BY data_venda;
    """)


    # ------------------------------------------------------
    # 4. CONFIRMAR A TRANSAÇÃO
    # ------------------------------------------------------

    conexao.commit()


    print("✅ Dimensão Data carregada com sucesso!")


except Exception as erro:

    # Caso alguma operação falhe,
    # desfaz as alterações realizadas.
    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro durante a carga da dim_data.")
    print("Tipo:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    # Fecha o cursor.
    if "cursor" in locals():
        cursor.close()

    # Fecha a conexão.
    if "conexao" in locals():
        conexao.close()

## Etapa 8 — Validação da Dimensão Data

Após carregar uma dimensão, precisamos validar se os registros foram criados corretamente.

Vamos verificar:

- quantidade de datas diferentes na Silver;
- quantidade de registros na dimensão;
- existência de datas duplicadas;
- primeiros registros da dimensão.

In [ ]:
# ==========================================================
# ETAPA 8 - VALIDAÇÃO DA DIMENSÃO DATA
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 1. DATAS DISTINTAS NA SILVER
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(DISTINCT data_venda)
        FROM silver.vendas;
    """)

    datas_silver = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 2. REGISTROS NA DIM_DATA
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM gold.dim_data;
    """)

    datas_gold = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 3. VERIFICAR DUPLICIDADES
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*) - COUNT(DISTINCT data)
        FROM gold.dim_data;
    """)

    datas_duplicadas = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 4. MOSTRAR RESULTADOS
    # ------------------------------------------------------

    print("=== VALIDAÇÃO DIM_DATA ===\n")

    print("Datas distintas na Silver:", datas_silver)
    print("Registros na dim_data:", datas_gold)
    print("Datas duplicadas:", datas_duplicadas)


    # ------------------------------------------------------
    # 5. VISUALIZAR ALGUNS REGISTROS
    # ------------------------------------------------------

    cursor.execute("""
        SELECT *
        FROM gold.dim_data
        ORDER BY data
        LIMIT 10;
    """)

    registros = cursor.fetchall()


    print("\n=== PRIMEIROS REGISTROS ===\n")


    for registro in registros:
        print(registro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 9 — Criação da Dimensão Produto

Agora criaremos nossa segunda dimensão:

`gold.dim_produto`

Na camada Silver, o nome do produto e sua categoria aparecem repetidamente em cada venda.

Na camada Gold criaremos uma dimensão contendo apenas os produtos distintos.

Cada produto receberá uma chave chamada:

`id_produto`

Essa chave será utilizada posteriormente pela tabela `fato_vendas`.

A estrutura será:

- `id_produto`: identificador único do produto;
- `produto`: nome do produto;
- `categoria`: categoria à qual o produto pertence.

In [ ]:
# ==========================================================
# ETAPA 9 - CRIAÇÃO DA DIMENSÃO PRODUTO
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS gold.dim_produto (

            id_produto SERIAL PRIMARY KEY,

            produto TEXT NOT NULL,

            categoria TEXT NOT NULL,

            UNIQUE (produto, categoria)
        );
    """)

    conexao.commit()

    print("✅ gold.dim_produto criada com sucesso!")


except Exception as erro:

    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro ao criar gold.dim_produto.")
    print("Tipo:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 10 — Carga da Dimensão Produto

Agora vamos carregar a `gold.dim_produto`.

A fonte dos dados será:

`silver.vendas`

Como o mesmo produto pode aparecer em centenas de vendas, utilizaremos `DISTINCT`.

Dessa maneira, cada combinação de produto e categoria aparecerá apenas uma vez na dimensão.

Fluxo:

`silver.vendas → produtos distintos → gold.dim_produto`

In [ ]:
# ==========================================================
# ETAPA 10 - CARGA DA DIMENSÃO PRODUTO
# ==========================================================

try:

    # ------------------------------------------------------
    # 1. ABRIR CONEXÃO
    # ------------------------------------------------------

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 2. LIMPAR A DIMENSÃO
    # ------------------------------------------------------

    # Remove registros existentes.
    #
    # RESTART IDENTITY também reinicia
    # a sequência do SERIAL.
    cursor.execute("""
        TRUNCATE TABLE gold.dim_produto
        RESTART IDENTITY;
    """)


    # ------------------------------------------------------
    # 3. CARREGAR OS PRODUTOS
    # ------------------------------------------------------

    cursor.execute("""
        INSERT INTO gold.dim_produto (
            produto,
            categoria
        )

        SELECT DISTINCT
            produto,
            categoria

        FROM silver.vendas

        WHERE
            produto IS NOT NULL
            AND categoria IS NOT NULL

        ORDER BY
            produto,
            categoria;
    """)


    # ------------------------------------------------------
    # 4. CONFIRMAR A TRANSAÇÃO
    # ------------------------------------------------------

    conexao.commit()


    print("✅ Dimensão Produto carregada com sucesso!")


except Exception as erro:

    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro durante a carga da dim_produto.")
    print("Tipo:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 11 — Validação da Dimensão Produto

Agora vamos verificar se a dimensão Produto foi carregada corretamente.

Vamos comparar:

- quantidade de produtos distintos existentes na Silver;
- quantidade de registros existentes na `dim_produto`;
- existência de duplicidades;
- primeiros registros da dimensão.

A quantidade de combinações distintas de produto e categoria na Silver deve ser igual à quantidade de registros da dimensão.

In [ ]:
# ==========================================================
# ETAPA 11 - VALIDAÇÃO DA DIMENSÃO PRODUTO
# ==========================================================

try:

    # ------------------------------------------------------
    # 1. ABRIR CONEXÃO
    # ------------------------------------------------------

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 2. PRODUTOS DISTINTOS NA SILVER
    # ------------------------------------------------------

    # A subconsulta identifica as combinações diferentes
    # de produto + categoria.
    #
    # Depois COUNT(*) conta essas combinações.
    cursor.execute("""
        SELECT COUNT(*)
        FROM (
            SELECT DISTINCT
                produto,
                categoria
            FROM silver.vendas
        ) AS produtos_distintos;
    """)

    produtos_silver = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 3. PRODUTOS NA GOLD
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM gold.dim_produto;
    """)

    produtos_gold = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 4. VERIFICAR DUPLICIDADES
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM (

            SELECT
                produto,
                categoria,
                COUNT(*)

            FROM gold.dim_produto

            GROUP BY
                produto,
                categoria

            HAVING COUNT(*) > 1

        ) AS duplicados;
    """)

    produtos_duplicados = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 5. MOSTRAR RESULTADO
    # ------------------------------------------------------

    print("=== VALIDAÇÃO DIM_PRODUTO ===\n")

    print("Produtos distintos na Silver:", produtos_silver)
    print("Produtos na dim_produto:", produtos_gold)
    print("Produtos duplicados:", produtos_duplicados)


    # ------------------------------------------------------
    # 6. MOSTRAR OS PRODUTOS
    # ------------------------------------------------------

    cursor.execute("""
        SELECT
            id_produto,
            produto,
            categoria

        FROM gold.dim_produto

        ORDER BY id_produto

        LIMIT 10;
    """)

    registros = cursor.fetchall()


    print("\n=== PRODUTOS CADASTRADOS ===\n")


    for registro in registros:
        print(registro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

In [ ]:
# ==========================================================
# VERIFICAR TABELAS EXISTENTES NO SCHEMA GOLD
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()

    # Consulta as tabelas existentes dentro
    # do schema gold.
    cursor.execute("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'gold'
        ORDER BY table_name;
    """)

    tabelas = cursor.fetchall()

    print("=== TABELAS EXISTENTES NA GOLD ===\n")

    for tabela in tabelas:
        print(tabela[0])


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 12 — Criação da Dimensão Cliente

Agora criaremos a terceira dimensão do nosso modelo:

`gold.dim_cliente`

Na tabela `silver.vendas`, o nome do cliente aparece repetidamente sempre que o mesmo cliente realiza uma nova compra.

Na camada Gold queremos armazenar cada cliente apenas uma vez.

Cada cliente receberá uma chave própria:

`id_cliente`

Essa chave será utilizada posteriormente na tabela `fato_vendas`.

A estrutura será:

- `id_cliente`: identificador único criado pelo PostgreSQL;
- `cliente`: nome do cliente.

Fluxo:

`silver.vendas → clientes distintos → gold.dim_cliente`

In [ ]:
# ==========================================================
# ETAPA 12 - CRIAÇÃO DA DIMENSÃO CLIENTE
# ==========================================================

try:

    # Abre a conexão com o PostgreSQL.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    # Cria o cursor para executar comandos SQL.
    cursor = conexao.cursor()


    # Cria a tabela dim_cliente dentro do schema gold.
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS gold.dim_cliente (

            -- Chave primária gerada automaticamente.
            id_cliente SERIAL PRIMARY KEY,

            -- Nome do cliente.
            cliente TEXT NOT NULL UNIQUE
        );
    """)


    # Confirma a criação da tabela.
    conexao.commit()

    print("✅ gold.dim_cliente criada com sucesso!")


except Exception as erro:

    # Se ocorrer algum erro, desfaz a transação.
    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro ao criar gold.dim_cliente.")
    print("Tipo:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 13 — Carga da Dimensão Cliente

Agora vamos carregar os clientes existentes na camada Silver para a dimensão Cliente.

Como um mesmo cliente pode realizar várias compras, seu nome pode aparecer diversas vezes em `silver.vendas`.

Utilizaremos `DISTINCT` para selecionar cada cliente apenas uma vez.

Exemplo:

Silver:

Maria
João
Maria
Carlos
João

Resultado na dimensão:

Maria
João
Carlos

In [ ]:
# ==========================================================
# ETAPA 13 - CARGA DA DIMENSÃO CLIENTE
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 1. LIMPAR A DIMENSÃO
    # ------------------------------------------------------

    # Remove registros existentes e reinicia
    # a sequência do id_cliente.
    cursor.execute("""
        TRUNCATE TABLE gold.dim_cliente
        RESTART IDENTITY;
    """)


    # ------------------------------------------------------
    # 2. INSERIR CLIENTES DISTINTOS
    # ------------------------------------------------------

    cursor.execute("""
        INSERT INTO gold.dim_cliente (
            cliente
        )

        SELECT DISTINCT
            cliente

        FROM silver.vendas

        WHERE cliente IS NOT NULL

        ORDER BY cliente;
    """)


    # ------------------------------------------------------
    # 3. CONFIRMAR A TRANSAÇÃO
    # ------------------------------------------------------

    conexao.commit()

    print("✅ Dimensão Cliente carregada com sucesso!")


except Exception as erro:

    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro durante a carga da dim_cliente.")
    print("Tipo:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 14 — Validação da Dimensão Cliente

Após realizar a carga, precisamos validar a dimensão.

Vamos comparar:

- quantidade de clientes distintos na Silver;
- quantidade de clientes existentes na Gold;
- quantidade de clientes duplicados.

Os dois primeiros valores devem ser iguais e a quantidade de duplicados deve ser zero.

In [ ]:
# ==========================================================
# ETAPA 14 - VALIDAÇÃO DA DIMENSÃO CLIENTE
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 1. CLIENTES DISTINTOS NA SILVER
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(DISTINCT cliente)
        FROM silver.vendas;
    """)

    clientes_silver = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 2. CLIENTES NA DIMENSÃO
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM gold.dim_cliente;
    """)

    clientes_gold = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 3. VERIFICAR DUPLICIDADES
    # ------------------------------------------------------

    # Como cliente possui UNIQUE, não deveria
    # existir nenhum cliente duplicado.
    cursor.execute("""
        SELECT COUNT(*) - COUNT(DISTINCT cliente)
        FROM gold.dim_cliente;
    """)

    clientes_duplicados = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 4. MOSTRAR RESULTADOS
    # ------------------------------------------------------

    print("=== VALIDAÇÃO DIM_CLIENTE ===\n")

    print("Clientes distintos na Silver:", clientes_silver)
    print("Clientes na dim_cliente:", clientes_gold)
    print("Clientes duplicados:", clientes_duplicados)


    # ------------------------------------------------------
    # 5. VISUALIZAR ALGUNS CLIENTES
    # ------------------------------------------------------

    cursor.execute("""
        SELECT
            id_cliente,
            cliente
        FROM gold.dim_cliente
        ORDER BY id_cliente
        LIMIT 10;
    """)

    registros = cursor.fetchall()


    print("\n=== PRIMEIROS CLIENTES ===\n")

    for registro in registros:
        print(registro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 15 — Criação da Dimensão Região

Agora criaremos a última dimensão do nosso modelo:

`gold.dim_regiao`

Na tabela `silver.vendas`, a região aparece repetidamente em diferentes vendas.

Na camada Gold queremos armazenar cada região apenas uma vez.

Cada região receberá uma chave própria:

`id_regiao`

Essa chave será utilizada posteriormente na tabela `fato_vendas`.

Fluxo:

`silver.vendas → regiões distintas → gold.dim_regiao`

In [ ]:
# ==========================================================
# ETAPA 15 - CRIAÇÃO DA DIMENSÃO REGIÃO
# ==========================================================

try:

    # ------------------------------------------------------
    # 1. ABRIR CONEXÃO
    # ------------------------------------------------------

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 2. CRIAR A DIMENSÃO
    # ------------------------------------------------------

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS gold.dim_regiao (

            -- Chave primária gerada automaticamente.
            id_regiao SERIAL PRIMARY KEY,

            -- Nome da região.
            --
            -- NOT NULL:
            -- a região precisa possuir um valor.
            --
            -- UNIQUE:
            -- impede que a mesma região seja
            -- cadastrada mais de uma vez.
            regiao TEXT NOT NULL UNIQUE
        );
    """)


    # ------------------------------------------------------
    # 3. CONFIRMAR A CRIAÇÃO
    # ------------------------------------------------------

    conexao.commit()

    print("✅ gold.dim_regiao criada com sucesso!")


except Exception as erro:

    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro ao criar gold.dim_regiao.")
    print("Tipo:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 19 — Construção da Tabela Fato

Agora criaremos a tabela central do nosso modelo dimensional:

`gold.fato_vendas`

Enquanto as dimensões descrevem características da venda, a tabela fato representa o evento que aconteceu.

Neste projeto, o evento é:

**uma venda realizada.**

A tabela fato armazenará:

### Chaves

- `id_venda`: identificador da venda;
- `id_data`: referência para `dim_data`;
- `id_produto`: referência para `dim_produto`;
- `id_cliente`: referência para `dim_cliente`;
- `id_regiao`: referência para `dim_regiao`.

### Métricas

- `quantidade`;
- `valor_unitario`;
- `desconto_percentual`;
- `valor_total`.

Também manteremos `forma_pagamento` como uma característica simples da transação.

As dimensões serão relacionadas à tabela fato através de **Foreign Keys (FK)**.

## Etapa 20 — Criação da Tabela Fato

Agora criaremos a estrutura física da `gold.fato_vendas`.

Diferentemente das dimensões, a tabela fato possuirá várias chaves estrangeiras.

Essas chaves estabelecerão os relacionamentos entre a venda e suas respectivas dimensões.

In [ ]:
# ==========================================================
# ETAPA 20 - CRIAÇÃO DA TABELA FATO_VENDAS
# ==========================================================

try:

    # ------------------------------------------------------
    # 1. ABRIR CONEXÃO
    # ------------------------------------------------------

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 2. CRIAR A TABELA FATO
    # ------------------------------------------------------

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS gold.fato_vendas (

            -- Identificador original da venda.
            id_venda INTEGER PRIMARY KEY,

            -- Chave da dimensão Data.
            id_data INTEGER NOT NULL,

            -- Chave da dimensão Produto.
            id_produto INTEGER NOT NULL,

            -- Chave da dimensão Cliente.
            id_cliente INTEGER NOT NULL,

            -- Chave da dimensão Região.
            id_regiao INTEGER NOT NULL,

            -- Métricas da venda.
            quantidade INTEGER NOT NULL,

            valor_unitario NUMERIC(12,2) NOT NULL,

            desconto_percentual INTEGER NOT NULL,

            valor_total NUMERIC(12,2) NOT NULL,

            -- Característica da transação.
            forma_pagamento TEXT NOT NULL,


            -- =============================================
            -- FOREIGN KEYS
            -- =============================================

            CONSTRAINT fk_data
                FOREIGN KEY (id_data)
                REFERENCES gold.dim_data(id_data),

            CONSTRAINT fk_produto
                FOREIGN KEY (id_produto)
                REFERENCES gold.dim_produto(id_produto),

            CONSTRAINT fk_cliente
                FOREIGN KEY (id_cliente)
                REFERENCES gold.dim_cliente(id_cliente),

            CONSTRAINT fk_regiao
                FOREIGN KEY (id_regiao)
                REFERENCES gold.dim_regiao(id_regiao)
        );
    """)


    # ------------------------------------------------------
    # 3. CONFIRMAR
    # ------------------------------------------------------

    conexao.commit()

    print("✅ gold.fato_vendas criada com sucesso!")


except Exception as erro:

    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro ao criar gold.fato_vendas.")
    print("Tipo:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

In [ ]:
# ==========================================================
# ETAPA 21 - VERIFICAR SE FATO_VENDAS FOI CRIADA
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # Procura a tabela no PostgreSQL.
    cursor.execute("""
        SELECT to_regclass('gold.fato_vendas');
    """)

    tabela = cursor.fetchone()[0]


    if tabela is not None:

        print("✅ Tabela encontrada:", tabela)

    else:

        print("❌ A tabela gold.fato_vendas não existe.")


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 22 — Relacionando Silver e Dimensões

A tabela `silver.vendas` ainda possui informações descritivas como:

- data da venda;
- nome do produto;
- nome do cliente;
- região.

Na tabela fato não queremos repetir essas informações.

Queremos armazenar as chaves das respectivas dimensões.

Para encontrar essas chaves utilizaremos `JOIN`.

Exemplo:

Se a Silver possui:

Produto = "Sofá"

E a `dim_produto` possui:

id_produto = 5  
produto = "Sofá"

O JOIN permitirá encontrar o `id_produto = 5`.

Esse processo será realizado para:

- Data → `id_data`
- Produto → `id_produto`
- Cliente → `id_cliente`
- Região → `id_regiao`

## Etapa 23 — Teste dos Relacionamentos

Antes de carregar a tabela fato, vamos testar os relacionamentos entre a Silver e nossas dimensões.

Utilizaremos `INNER JOIN`.

O objetivo é verificar se conseguimos encontrar corretamente as chaves correspondentes a cada venda.

Neste momento nenhum dado será inserido.

Executaremos apenas uma consulta de validação.

In [ ]:
# ==========================================================
# ETAPA 23 - TESTE DOS JOINS
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # CONSULTAR SILVER + DIMENSÕES
    # ------------------------------------------------------

    cursor.execute("""
        SELECT

            -- Identificador original da venda.
            s.id_venda,


            -- DATA
            s.data_venda,
            d.id_data,


            -- PRODUTO
            s.produto,
            p.id_produto,


            -- CLIENTE
            s.cliente,
            c.id_cliente,


            -- REGIÃO
            s.regiao,
            r.id_regiao,


            -- MÉTRICAS
            s.quantidade,
            s.valor_total

        FROM silver.vendas AS s


        -- =================================================
        -- RELACIONAMENTO COM DIM_DATA
        -- =================================================

        INNER JOIN gold.dim_data AS d

            ON s.data_venda = d.data


        -- =================================================
        -- RELACIONAMENTO COM DIM_PRODUTO
        -- =================================================

        INNER JOIN gold.dim_produto AS p

            ON s.produto = p.produto

            AND s.categoria = p.categoria


        -- =================================================
        -- RELACIONAMENTO COM DIM_CLIENTE
        -- =================================================

        INNER JOIN gold.dim_cliente AS c

            ON s.cliente = c.cliente


        -- =================================================
        -- RELACIONAMENTO COM DIM_REGIAO
        -- =================================================

        INNER JOIN gold.dim_regiao AS r

            ON s.regiao = r.regiao


        ORDER BY s.id_venda

        LIMIT 10;
    """)


    registros = cursor.fetchall()


    print("=== TESTE DOS RELACIONAMENTOS ===\n")


    for registro in registros:
        print(registro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

In [ ]:
# ==========================================================
# ETAPA 24 - VALIDAR OS RELACIONAMENTOS
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # QUANTIDADE NA SILVER
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.vendas;
    """)

    total_silver = cursor.fetchone()[0]


    # ------------------------------------------------------
    # QUANTIDADE APÓS OS JOINS
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)

        FROM silver.vendas AS s

        INNER JOIN gold.dim_data AS d
            ON s.data_venda = d.data

        INNER JOIN gold.dim_produto AS p
            ON s.produto = p.produto
            AND s.categoria = p.categoria

        INNER JOIN gold.dim_cliente AS c
            ON s.cliente = c.cliente

        INNER JOIN gold.dim_regiao AS r
            ON s.regiao = r.regiao;
    """)

    total_join = cursor.fetchone()[0]


    # ------------------------------------------------------
    # RESULTADO
    # ------------------------------------------------------

    print("=== VALIDAÇÃO DOS RELACIONAMENTOS ===\n")

    print("Registros na Silver:", total_silver)
    print("Registros encontrados pelos JOINs:", total_join)


    if total_silver == total_join:

        print("\n✅ Todos os registros encontraram suas dimensões.")

    else:

        diferenca = total_silver - total_join

        print("\n❌ Existem registros sem correspondência.")
        print("Quantidade não encontrada:", diferenca)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

In [ ]:
# ==========================================================
# ETAPA 24.1 - DIAGNÓSTICO DAS DIMENSÕES
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # CONTAR REGISTROS DE CADA TABELA
    # ------------------------------------------------------

    tabelas = [
        "silver.vendas",
        "gold.dim_data",
        "gold.dim_produto",
        "gold.dim_cliente",
        "gold.dim_regiao"
    ]


    print("=== QUANTIDADE DE REGISTROS ===\n")


    # Percorre nossa lista de tabelas.
    for tabela in tabelas:

        cursor.execute(f"""
            SELECT COUNT(*)
            FROM {tabela};
        """)

        quantidade = cursor.fetchone()[0]

        print(f"{tabela}: {quantidade}")


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

In [ ]:
# ==========================================================
# ETAPA 24.2 - DIAGNÓSTICO DOS JOINS
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    print("=== DIAGNÓSTICO DOS JOINS ===\n")


    # ------------------------------------------------------
    # 1. TESTAR DIM_DATA
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.vendas AS s

        INNER JOIN gold.dim_data AS d
            ON s.data_venda = d.data;
    """)

    join_data = cursor.fetchone()[0]

    print("JOIN dim_data:", join_data)


    # ------------------------------------------------------
    # 2. TESTAR DIM_PRODUTO
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.vendas AS s

        INNER JOIN gold.dim_produto AS p
            ON s.produto = p.produto
            AND s.categoria = p.categoria;
    """)

    join_produto = cursor.fetchone()[0]

    print("JOIN dim_produto:", join_produto)


    # ------------------------------------------------------
    # 3. TESTAR DIM_CLIENTE
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.vendas AS s

        INNER JOIN gold.dim_cliente AS c
            ON s.cliente = c.cliente;
    """)

    join_cliente = cursor.fetchone()[0]

    print("JOIN dim_cliente:", join_cliente)


    # ------------------------------------------------------
    # 4. TESTAR DIM_REGIAO
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.vendas AS s

        INNER JOIN gold.dim_regiao AS r
            ON s.regiao = r.regiao;
    """)

    join_regiao = cursor.fetchone()[0]

    print("JOIN dim_regiao:", join_regiao)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

In [ ]:
# ==========================================================
# CARGA DA DIMENSÃO CLIENTE
# ==========================================================

try:

    # ------------------------------------------------------
    # 1. ABRIR CONEXÃO
    # ------------------------------------------------------

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 2. CARREGAR CLIENTES
    # ------------------------------------------------------

    # Selecionamos os clientes distintos existentes
    # na camada Silver.
    #
    # ON CONFLICT DO NOTHING:
    # caso algum cliente já exista na dimensão,
    # o PostgreSQL simplesmente não tenta inseri-lo novamente.
    cursor.execute("""
        INSERT INTO gold.dim_cliente (
            cliente
        )

        SELECT DISTINCT
            cliente

        FROM silver.vendas

        WHERE cliente IS NOT NULL

        ON CONFLICT (cliente) DO NOTHING;
    """)


    # ------------------------------------------------------
    # 3. CONFIRMAR A TRANSAÇÃO
    # ------------------------------------------------------

    conexao.commit()


    # ------------------------------------------------------
    # 4. VALIDAR A CARGA
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM gold.dim_cliente;
    """)

    total_clientes = cursor.fetchone()[0]


    print("✅ Dimensão Cliente carregada com sucesso!")
    print("Clientes existentes na dimensão:", total_clientes)


except Exception as erro:

    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro ao carregar dim_cliente.")
    print("Tipo:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

In [ ]:
# ==========================================================
# CARGA DA DIMENSÃO REGIÃO
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # CARREGAR REGIÕES
    # ------------------------------------------------------

    cursor.execute("""
        INSERT INTO gold.dim_regiao (
            regiao
        )

        SELECT DISTINCT
            regiao

        FROM silver.vendas

        WHERE regiao IS NOT NULL

        ON CONFLICT (regiao) DO NOTHING;
    """)


    conexao.commit()


    # ------------------------------------------------------
    # VALIDAR
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM gold.dim_regiao;
    """)

    total_regioes = cursor.fetchone()[0]


    print("✅ Dimensão Região carregada com sucesso!")
    print("Regiões existentes na dimensão:", total_regioes)


except Exception as erro:

    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro ao carregar dim_regiao.")
    print("Tipo:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

In [ ]:
# ==========================================================
# ETAPA 24 - VALIDAÇÃO DOS RELACIONAMENTOS
# ==========================================================

try:

    # Abre conexão com o PostgreSQL.
    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 1. CONTAR REGISTROS DA SILVER
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM silver.vendas;
    """)

    total_silver = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 2. REALIZAR TODOS OS JOINS
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)

        FROM silver.vendas AS s

        INNER JOIN gold.dim_data AS d
            ON s.data_venda = d.data

        INNER JOIN gold.dim_produto AS p
            ON s.produto = p.produto
            AND s.categoria = p.categoria

        INNER JOIN gold.dim_cliente AS c
            ON s.cliente = c.cliente

        INNER JOIN gold.dim_regiao AS r
            ON s.regiao = r.regiao;
    """)

    total_join = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 3. COMPARAR OS RESULTADOS
    # ------------------------------------------------------

    print("=== VALIDAÇÃO DOS RELACIONAMENTOS ===\n")

    print("Registros na Silver:", total_silver)
    print("Registros encontrados pelos JOINs:", total_join)


    if total_silver == total_join:

        print("\n✅ Todos os registros encontraram suas dimensões.")
        print("✅ A carga da fato_vendas pode prosseguir.")

    else:

        diferenca = total_silver - total_join

        print("\n❌ Existem registros sem correspondência.")
        print("Quantidade não encontrada:", diferenca)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 25 — Carga da Tabela Fato

Após validar que todos os registros da Silver encontraram correspondência nas dimensões, podemos carregar a tabela `gold.fato_vendas`.

Nesta etapa utilizaremos `JOIN` para substituir os valores descritivos pelas respectivas chaves das dimensões.

Exemplo:

Silver:

- Data: 10/01/2026
- Produto: Sofá
- Cliente: Maria
- Região: Nordeste

Gold:

- id_data: 10
- id_produto: 5
- id_cliente: 7
- id_regiao: 2

As métricas da venda serão mantidas na tabela fato.

Dessa forma completaremos nosso Star Schema.

In [ ]:
# ==========================================================
# ETAPA 25 - CARGA DA TABELA FATO_VENDAS
# ==========================================================

try:

    # ------------------------------------------------------
    # 1. ABRIR CONEXÃO
    # ------------------------------------------------------

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 2. LIMPAR A TABELA FATO
    # ------------------------------------------------------

    # Diferentemente das dimensões, podemos limpar
    # a fato porque nenhuma outra tabela do nosso
    # modelo depende dela neste momento.
    #
    # Isso permite executar novamente esta etapa
    # sem duplicar as vendas.
    cursor.execute("""
        TRUNCATE TABLE gold.fato_vendas;
    """)


    # ------------------------------------------------------
    # 3. INSERIR AS VENDAS
    # ------------------------------------------------------

    cursor.execute("""
        INSERT INTO gold.fato_vendas (

            id_venda,
            id_data,
            id_produto,
            id_cliente,
            id_regiao,
            quantidade,
            valor_unitario,
            desconto_percentual,
            valor_total,
            forma_pagamento
        )

        SELECT

            -- Identificador original da venda.
            s.id_venda,


            -- Chave encontrada na dimensão Data.
            d.id_data,


            -- Chave encontrada na dimensão Produto.
            p.id_produto,


            -- Chave encontrada na dimensão Cliente.
            c.id_cliente,


            -- Chave encontrada na dimensão Região.
            r.id_regiao,


            -- Métricas provenientes da Silver.
            s.quantidade,

            s.valor_unitario,

            s.desconto_percentual,

            s.valor_total,


            -- Característica da transação.
            s.forma_pagamento


        FROM silver.vendas AS s


        -- =================================================
        -- DIMENSÃO DATA
        -- =================================================

        INNER JOIN gold.dim_data AS d
            ON s.data_venda = d.data


        -- =================================================
        -- DIMENSÃO PRODUTO
        -- =================================================

        INNER JOIN gold.dim_produto AS p
            ON s.produto = p.produto
            AND s.categoria = p.categoria


        -- =================================================
        -- DIMENSÃO CLIENTE
        -- =================================================

        INNER JOIN gold.dim_cliente AS c
            ON s.cliente = c.cliente


        -- =================================================
        -- DIMENSÃO REGIÃO
        -- =================================================

        INNER JOIN gold.dim_regiao AS r
            ON s.regiao = r.regiao;
    """)


    # ------------------------------------------------------
    # 4. CONFIRMAR A TRANSAÇÃO
    # ------------------------------------------------------

    conexao.commit()


    # ------------------------------------------------------
    # 5. CONTAR OS REGISTROS INSERIDOS
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM gold.fato_vendas;
    """)

    total_fato = cursor.fetchone()[0]


    print("✅ Carga da fato_vendas concluída!")
    print("Registros inseridos:", total_fato)


except Exception as erro:

    # Caso alguma parte da carga falhe,
    # desfazemos a transação.
    if "conexao" in locals():
        conexao.rollback()

    print("❌ Erro durante a carga da fato_vendas.")
    print("Tipo:", type(erro).__name__)
    print("Detalhes:", erro)


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()

## Etapa 26 — Validação Final da Camada Gold

A tabela fato foi carregada.

Agora realizaremos uma validação final para verificar:

- quantidade de vendas na Silver;
- quantidade de vendas na Gold;
- quantidade de IDs de venda distintos;
- faturamento da Silver;
- faturamento da Gold;
- existência de chaves estrangeiras nulas.

Essa etapa permitirá verificar se nossa transformação manteve a integridade dos dados.

In [ ]:
# ==========================================================
# ETAPA 26 - VALIDAÇÃO FINAL DA GOLD
# ==========================================================

try:

    conexao = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )

    cursor = conexao.cursor()


    # ------------------------------------------------------
    # 1. VALIDAR A SILVER
    # ------------------------------------------------------

    cursor.execute("""
        SELECT
            COUNT(*),
            COUNT(DISTINCT id_venda),
            SUM(valor_total)
        FROM silver.vendas;
    """)

    resultado_silver = cursor.fetchone()

    registros_silver = resultado_silver[0]
    ids_silver = resultado_silver[1]
    faturamento_silver = resultado_silver[2]


    # ------------------------------------------------------
    # 2. VALIDAR A GOLD
    # ------------------------------------------------------

    cursor.execute("""
        SELECT
            COUNT(*),
            COUNT(DISTINCT id_venda),
            SUM(valor_total)
        FROM gold.fato_vendas;
    """)

    resultado_gold = cursor.fetchone()

    registros_gold = resultado_gold[0]
    ids_gold = resultado_gold[1]
    faturamento_gold = resultado_gold[2]


    # ------------------------------------------------------
    # 3. VERIFICAR CHAVES NULAS
    # ------------------------------------------------------

    cursor.execute("""
        SELECT COUNT(*)
        FROM gold.fato_vendas

        WHERE
            id_data IS NULL
            OR id_produto IS NULL
            OR id_cliente IS NULL
            OR id_regiao IS NULL;
    """)

    chaves_nulas = cursor.fetchone()[0]


    # ------------------------------------------------------
    # 4. EXIBIR O RESULTADO
    # ------------------------------------------------------

    print("=== VALIDAÇÃO FINAL DA GOLD ===\n")

    print("Registros Silver:", registros_silver)
    print("Registros Gold:", registros_gold)

    print("\nIDs únicos Silver:", ids_silver)
    print("IDs únicos Gold:", ids_gold)

    print("\nFaturamento Silver:", faturamento_silver)
    print("Faturamento Gold:", faturamento_gold)

    print("\nChaves estrangeiras nulas:", chaves_nulas)


    # ------------------------------------------------------
    # 5. RESULTADO DA VALIDAÇÃO
    # ------------------------------------------------------

    if (
        registros_silver == registros_gold
        and ids_silver == ids_gold
        and faturamento_silver == faturamento_gold
        and chaves_nulas == 0
    ):

        print("\n✅ CAMADA GOLD VALIDADA COM SUCESSO!")

    else:

        print("\n❌ Existem diferenças entre Silver e Gold.")


finally:

    if "cursor" in locals():
        cursor.close()

    if "conexao" in locals():
        conexao.close()